# Land Patches – Experimente (MLP, CNN, Augmentare, Fine-tuning din Imagebits)



In [ ]:
# ==========================================
# 1. SETUP & CONFIG
# ==========================================
import os
import random
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

from sklearn.metrics import confusion_matrix, accuracy_score, f1_score, classification_report

from IPython.display import display

%matplotlib inline

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

@dataclass
class CFG:
    DATA_DIR: str = "land_patches"
    IMG_SIZE: int = 64
    BATCH_SIZE: int = 64
    EPOCHS: int = 20
    LR: float = 1e-3
    WEIGHT_DECAY: float = 1e-4
    DEVICE: str = "cuda" if torch.cuda.is_available() else "cpu"
    NUM_WORKERS: int = 2

print("Device:", CFG.DEVICE)


In [ ]:
# ==========================================
# 2. DATASET CHECK + EDA
# ==========================================
train_dir = Path(CFG.DATA_DIR) / "train"
val_dir   = Path(CFG.DATA_DIR) / "val"
test_dir  = Path(CFG.DATA_DIR) / "test"

for d in [train_dir, val_dir, test_dir]:
    print(d, "exists:", d.exists())

assert train_dir.exists() and val_dir.exists() and test_dir.exists(), (
    "Nu găsesc structura train/val/test. Verifică CFG.DATA_DIR."
)

def count_images_per_class(split_dir: Path):
    counts = {}
    for cls in sorted([p.name for p in split_dir.iterdir() if p.is_dir()]):
        counts[cls] = len(list((split_dir/cls).rglob("*.jpg"))) + len(list((split_dir/cls).rglob("*.png"))) + len(list((split_dir/cls).rglob("*.jpeg")))
    return counts

train_counts = count_images_per_class(train_dir)
val_counts   = count_images_per_class(val_dir)
test_counts  = count_images_per_class(test_dir)

df_counts = pd.DataFrame({"train": train_counts, "val": val_counts, "test": test_counts}).fillna(0).astype(int)
display(df_counts)

plt.figure(figsize=(10,4))
df_counts["train"].plot(kind="bar")
plt.title("Land patches – distribuția claselor (train)")
plt.ylabel("# imagini")
plt.show()

# Exemple vizuale pe câteva clase
def show_samples(split_dir: Path, n_classes=5, n_per_class=3):
    classes = sorted([p.name for p in split_dir.iterdir() if p.is_dir()])[:n_classes]
    plt.figure(figsize=(n_per_class*3, n_classes*3))
    idx = 1
    for cls in classes:
        imgs = list((split_dir/cls).glob("*.jpg")) + list((split_dir/cls).glob("*.png")) + list((split_dir/cls).glob("*.jpeg"))
        imgs = imgs[:n_per_class]
        for img_path in imgs:
            img = Image.open(img_path).convert("RGB")
            plt.subplot(n_classes, n_per_class, idx)
            plt.imshow(img)
            plt.axis("off")
            plt.title(cls, fontsize=9)
            idx += 1
    plt.tight_layout()
    plt.show()

show_samples(train_dir, n_classes=5, n_per_class=3)


In [ ]:
# ==========================================
# 3. TRANSFORMS + DATALOADERS
# ==========================================
# Eval transform (val/test): FĂRĂ augmentări
eval_tfms = transforms.Compose([
    transforms.Resize((CFG.IMG_SIZE, CFG.IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Train baseline: fără augmentări
train_tfms_base = transforms.Compose([
    transforms.Resize((CFG.IMG_SIZE, CFG.IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Train augmentat (potrivit pentru satelit)
train_tfms_aug = transforms.Compose([
    transforms.Resize((CFG.IMG_SIZE, CFG.IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def make_loaders(train_tfms):
    train_ds = datasets.ImageFolder(root=str(train_dir), transform=train_tfms)
    val_ds   = datasets.ImageFolder(root=str(val_dir), transform=eval_tfms)
    test_ds  = datasets.ImageFolder(root=str(test_dir), transform=eval_tfms)

    train_loader = DataLoader(train_ds, batch_size=CFG.BATCH_SIZE, shuffle=True, num_workers=CFG.NUM_WORKERS)
    val_loader   = DataLoader(val_ds, batch_size=CFG.BATCH_SIZE, shuffle=False, num_workers=CFG.NUM_WORKERS)
    test_loader  = DataLoader(test_ds, batch_size=CFG.BATCH_SIZE, shuffle=False, num_workers=CFG.NUM_WORKERS)
    return train_loader, val_loader, test_loader, train_ds.classes

train_loader_base, val_loader, test_loader, class_names = make_loaders(train_tfms_base)
train_loader_aug, _, _, _ = make_loaders(train_tfms_aug)

print("Classes:", class_names)


In [ ]:
# ==========================================
# 4. MODELE (MLP + CNN pentru 64×64)
# ==========================================
class SimpleMLP(nn.Module):
    def __init__(self, img_size=CFG.IMG_SIZE, num_classes=10):
        super().__init__()
        input_size = img_size * img_size * 3
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(input_size, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes),
        )
    def forward(self, x):
        return self.net(x)

class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),   # 64 -> 32

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),   # 32 -> 16

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),   # 16 -> 8
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 8 * 8, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes),
        )
    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


In [ ]:
# ==========================================
# 5. TRAIN / EVAL UTILS
# ==========================================
def train_model(model, train_loader, val_loader, optimizer, criterion, epochs, device):
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    model.to(device)

    for epoch in range(epochs):
        model.train()
        tr_loss, tr_correct, tr_total = 0.0, 0, 0

        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()

            tr_loss += loss.item() * x.size(0)
            tr_total += y.size(0)
            tr_correct += (out.argmax(1) == y).sum().item()

        train_loss = tr_loss / tr_total
        train_acc  = tr_correct / tr_total

        model.eval()
        va_loss, va_correct, va_total = 0.0, 0, 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                out = model(x)
                loss = criterion(out, y)
                va_loss += loss.item() * x.size(0)
                va_total += y.size(0)
                va_correct += (out.argmax(1) == y).sum().item()

        val_loss = va_loss / va_total
        val_acc  = va_correct / va_total

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)

        print(f"Epoch {epoch+1}/{epochs} | "
              f"Train Loss {train_loss:.4f} Acc {train_acc:.4f} | "
              f"Val Loss {val_loss:.4f} Acc {val_acc:.4f}")
    return history

def plot_history(history, title="Training"):
    epochs = range(1, len(history['train_loss'])+1)

    # Loss (train vs val) – același grafic
    plt.figure(figsize=(10,4))
    plt.plot(epochs, history['train_loss'], label="Train Loss")
    plt.plot(epochs, history['val_loss'], label="Val Loss")
    plt.title(title + " - Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.tight_layout()
    plt.show()

    # Metrică (Accuracy) – alt grafic separat
    plt.figure(figsize=(10,4))
    plt.plot(epochs, history['train_acc'], label="Train Accuracy")
    plt.plot(epochs, history['val_acc'], label="Val Accuracy")
    plt.title(title + " - Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.tight_layout()
    plt.show()

def plot_compare(hist_a, hist_b, label_a="No Aug", label_b="Aug", title="CNN"):
    epochs = range(1, len(hist_a['val_loss'])+1)

    plt.figure(figsize=(10,4))
    plt.plot(epochs, hist_a['val_loss'], label=f"{label_a} - Val Loss")
    plt.plot(epochs, hist_b['val_loss'], label=f"{label_b} - Val Loss")
    plt.title(title + " | Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.show()

    plt.figure(figsize=(10,4))
    plt.plot(epochs, hist_a['val_acc'], label=f"{label_a} - Val Acc")
    plt.plot(epochs, hist_b['val_acc'], label=f"{label_b} - Val Acc")
    plt.title(title + " | Validation Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.show()

def eval_metrics(model, loader, device, class_names, title="Eval"):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            out = model(x)
            preds = out.argmax(1).cpu().numpy().tolist()
            all_preds.extend(preds)
            all_labels.extend(y.numpy().tolist())

    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds, average="macro")

    print(f"{title}: acc={acc:.4f}, macroF1={f1:.4f}")
    print("\nClassification report:")
    print(classification_report(all_labels, all_preds, target_names=class_names, digits=4))

    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(10,8))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f"{title} - Confusion Matrix\nAcc={acc:.4f}, F1={f1:.4f}")
    plt.ylabel("True")
    plt.xlabel("Pred")
    plt.show()

    return acc, f1


In [ ]:
# ==========================================
# 6. EXP 1: MLP Baseline
# ==========================================
criterion = nn.CrossEntropyLoss()

mlp = SimpleMLP(img_size=CFG.IMG_SIZE, num_classes=len(class_names)).to(CFG.DEVICE)
opt = optim.Adam(mlp.parameters(), lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY)

hist_mlp = train_model(mlp, train_loader_base, val_loader, opt, criterion, CFG.EPOCHS, CFG.DEVICE)
plot_history(hist_mlp, "MLP Baseline")

mlp_val_acc = max(hist_mlp['val_acc'])
mlp_test_acc, mlp_test_f1 = eval_metrics(mlp, test_loader, CFG.DEVICE, class_names, title="MLP Test")


In [ ]:
# ==========================================
# 7. EXP 2A: CNN Baseline (No Aug)
# ==========================================
cnn_base = SimpleCNN(num_classes=len(class_names)).to(CFG.DEVICE)
opt = optim.Adam(cnn_base.parameters(), lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY)

hist_cnn_base = train_model(cnn_base, train_loader_base, val_loader, opt, criterion, CFG.EPOCHS, CFG.DEVICE)
plot_history(hist_cnn_base, "CNN Baseline (No Aug)")

cnn_base_best_val = max(hist_cnn_base['val_acc'])
cnn_base_test_acc, cnn_base_test_f1 = eval_metrics(cnn_base, test_loader, CFG.DEVICE, class_names, title="CNN Baseline Test")


In [ ]:
# ==========================================
# 8. EXP 2B: CNN + Augmentare
# ==========================================
cnn_aug = SimpleCNN(num_classes=len(class_names)).to(CFG.DEVICE)
opt = optim.Adam(cnn_aug.parameters(), lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY)

hist_cnn_aug = train_model(cnn_aug, train_loader_aug, val_loader, opt, criterion, CFG.EPOCHS, CFG.DEVICE)
plot_history(hist_cnn_aug, "CNN Augmented")

cnn_aug_best_val = max(hist_cnn_aug['val_acc'])
cnn_aug_test_acc, cnn_aug_test_f1 = eval_metrics(cnn_aug, test_loader, CFG.DEVICE, class_names, title="CNN Augmented Test")

plot_compare(hist_cnn_base, hist_cnn_aug, label_a="No Aug", label_b="Aug", title="CNN: No Aug vs Aug")


In [ ]:
# ==========================================
# 9. EXP 3: Fine-tuning Imagebits -> Land patches
# ==========================================

import os
import torch
import torch.nn as nn
import torch.optim as optim

IMAGEBITS_CKPT = "cnn_imagebits_64.pth"

if not os.path.exists(IMAGEBITS_CKPT):
    print(f"Checkpoint not found: {IMAGEBITS_CKPT}. Salvează-l din notebook-ul Imagebits și revino.")
else:
    ckpt = torch.load(IMAGEBITS_CKPT, map_location=CFG.DEVICE)

    ft_model = SimpleCNN(num_classes=len(class_names)).to(CFG.DEVICE)

    # --- 1) Încarcă backbone-ul (features) din checkpoint, ignorând classifier-ul ---
    state = ckpt.get("state_dict", ckpt) 
    filtered = {k: v for k, v in state.items() if k.startswith("features.")}

    incompatible = ft_model.load_state_dict(filtered, strict=False)

    if hasattr(incompatible, "missing_keys") and hasattr(incompatible, "unexpected_keys"):
        print("Loaded features from checkpoint.")
        if incompatible.unexpected_keys:
            print("Unexpected keys (ignored):", incompatible.unexpected_keys[:10])
        if incompatible.missing_keys:
            print("Missing keys (expected, mostly classifier):", incompatible.missing_keys[:10])

    # --- 2) Reinițializează head-ul pentru Land patches---
    ft_model.classifier[-1] = nn.Linear(ft_model.classifier[-1].in_features, len(class_names)).to(CFG.DEVICE)

    # --- 3) Fine-tuning faza 1: freeze backbone, antrenezi doar head-ul ---
    for p in ft_model.features.parameters():
        p.requires_grad = False
    for p in ft_model.classifier.parameters():
        p.requires_grad = True

    opt = optim.Adam(ft_model.classifier.parameters(), lr=1e-3, weight_decay=CFG.WEIGHT_DECAY)

    print("Fine-tuning phase 1 (head only)...")
    hist_ft_head = train_model(
        ft_model, train_loader_aug, val_loader,
        opt, criterion,
        epochs=5,
        device=CFG.DEVICE
    )

    # --- 4) Fine-tuning faza 2: unfreeze tot, LR mai mic pe tot modelul ---
    for p in ft_model.features.parameters():
        p.requires_grad = True

    opt = optim.Adam(ft_model.parameters(), lr=1e-4, weight_decay=CFG.WEIGHT_DECAY)

    print("Fine-tuning phase 2 (full model)...")
    hist_ft_full = train_model(
        ft_model, train_loader_aug, val_loader,
        opt, criterion,
        epochs=10,
        device=CFG.DEVICE
    )

    # --- 5) Combină history pentru plot ---
    hist_ft = {
        "train_loss": hist_ft_head["train_loss"] + hist_ft_full["train_loss"],
        "val_loss":   hist_ft_head["val_loss"]   + hist_ft_full["val_loss"],
        "train_acc":  hist_ft_head["train_acc"]  + hist_ft_full["train_acc"],
        "val_acc":    hist_ft_head["val_acc"]    + hist_ft_full["val_acc"],
    }
    plot_history(hist_ft, "CNN Fine-tuned (Imagebits -> Land patches)")

    ft_best_val = max(hist_ft["val_acc"])
    ft_test_acc, ft_test_f1 = eval_metrics(
        ft_model, test_loader, CFG.DEVICE, class_names, title="Fine-tuned CNN Test"
    )


In [ ]:
# ==========================================
# 10. TABEL COMPARATIV (setup + metrice)
# ==========================================
rows = []

rows.append({
    "Model": "MLP baseline",
    "Arch details": "Flatten -> Linear(512) -> ReLU -> Dropout(0.3) -> Linear(128) -> ReLU -> Dropout(0.2) -> Linear(10)",
    "Dropout": "0.3/0.2",
    "Arch": "MLP",
    "Aug": "no",
    "Optimizer": "Adam",
    "LR": CFG.LR,
    "Weight decay": CFG.WEIGHT_DECAY,
    "Batch": CFG.BATCH_SIZE,
    "Epochs": CFG.EPOCHS,
    "Best Val Acc": float(mlp_val_acc),
    "Test Acc": float(mlp_test_acc),
    "Test Macro-F1": float(mlp_test_f1),
})

rows.append({
    "Model": "CNN baseline (no aug)",
    "Arch details": "Conv(3->32)->BN->ReLU->Pool; Conv(32->64)->BN->ReLU->Pool; Conv(64->128)->BN->ReLU->Pool; FC 256; FC 10",
    "Dropout": "0.4",
    "Arch": "CNN",
    "Aug": "no",
    "Optimizer": "Adam",
    "LR": CFG.LR,
    "Weight decay": CFG.WEIGHT_DECAY,
    "Batch": CFG.BATCH_SIZE,
    "Epochs": CFG.EPOCHS,
    "Best Val Acc": float(cnn_base_best_val),
    "Test Acc": float(cnn_base_test_acc),
    "Test Macro-F1": float(cnn_base_test_f1),
})

rows.append({
    "Model": "CNN augmented",
    "Arch details": "Same CNN; train aug: HFlip/VFlip/Rot(10)/ColorJitter",
    "Dropout": "0.4",
    "Arch": "CNN",
    "Aug": "yes",
    "Optimizer": "Adam",
    "LR": CFG.LR,
    "Weight decay": CFG.WEIGHT_DECAY,
    "Batch": CFG.BATCH_SIZE,
    "Epochs": CFG.EPOCHS,
    "Best Val Acc": float(cnn_aug_best_val),
    "Test Acc": float(cnn_aug_test_acc),
    "Test Macro-F1": float(cnn_aug_test_f1),
})

if "ft_best_val" in globals():
    rows.append({
        "Model": "CNN fine-tuned (from Imagebits)",
        "Arch details": "Same CNN; init from Imagebits features; head reinit; freeze->unfreeze",
        "Dropout": "0.4",
        "Arch": "CNN",
        "Aug": "yes",
        "Optimizer": "Adam (head lr=1e-3; full lr=1e-4)",
        "LR": "1e-3/1e-4",
        "Weight decay": CFG.WEIGHT_DECAY,
        "Batch": CFG.BATCH_SIZE,
        "Epochs": "5 + 10",
        "Best Val Acc": float(ft_best_val),
        "Test Acc": float(ft_test_acc),
        "Test Macro-F1": float(ft_test_f1),
    })

df_results = pd.DataFrame(rows)
display(df_results)
